# Yahtzee state explorer

Run this notebook from your Yahtzee project, usually from the `math/` directory
next to files like `value_iteration.py`, `precomputed.py`, `constants.py`,
`state_properties.py`, and the `data/` folder.

It assumes you have already run value iteration and have files like:

`data/state_properties/level_kk/<13-bit mask>.npz`

The original helpers below let you inspect a reduced game state, a roll, the
optimal first keep, second keep, final category choice, and nearby alternatives.

Additional helpers now inspect the aggregate state properties we computed:

- `reach_prob`
- `expected_score_before`
- `expected_score_after_check`
- `p_top_bonus_after`
- `score_dist_before`
- `score_dist_after`
- `box_score_dist_before_XX`
- `box_score_dist_after_XX`


In [1]:
from pathlib import Path
import os, sys

def find_math_dir():
    """Find the directory containing the Yahtzee math Python files."""
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd / "math",
        cwd.parent / "math",
    ]
    for candidate in candidates:
        if (candidate / "constants.py").exists() and (candidate / "state_properties.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the Yahtzee math directory. "
        "Open this notebook from the project root, math/, or math/notebooks/."
    )

MATH_DIR = find_math_dir()
os.chdir(MATH_DIR)

if str(MATH_DIR) not in sys.path:
    sys.path.insert(0, str(MATH_DIR))

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 252)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

from constants import *
from precomputed import (
    ALL_DICE_STATES, ALL_DICE_FREQS, ALL_KEEPS, KEEP_IDX, KEEPS_FOR_DICE,
    REROLL_OUTCOMES, DICE_IDX,
    dice_values_to_idx, dice_idx_to_values, dice_idx_to_vec, dice_vec_to_idx,
    SCORE_ROWS, JOKER_SCORE_ROWS, IS_YAHTZEE_T, YAHTZEE_FACE_T,
)
from reduced_game_state import ReducedGameState
from state_properties import STATE_PROPERTIES_DIR, shard_path, load_shard, row_index
from value_iteration import load_V_next, REROLL_MATRIX, REROLL_OFFSETS, REROLL_PAIR_KEEPS

MATH_DIR


PosixPath('/Users/patrickliscio/My Documents/Pycharm Projects/Ballpark-Figures/yahtzee/math')

## Basic display helpers

In [2]:
def cat_name(c):
    return CATEGORY_NAMES[int(c)]

def mask_from_categories(categories):
    '''
    categories can be ints or names from CATEGORY_NAMES.
    Example:
        mask_from_categories(["Ones", "Twos", YAHTZEE])
    '''
    mask = 0
    for c in categories:
        if isinstance(c, str):
            c = CATEGORY_NAMES.index(c)
        mask |= 1 << int(c)
    return mask

def categories_from_mask(mask):
    return [CATEGORY_NAMES[c] for c in range(NUM_CATEGORIES) if mask & (1 << c)]

def keep_to_values(keep_idx):
    keep = ALL_KEEPS[int(keep_idx)]
    return tuple(face for face, count in enumerate(keep, start=1) for _ in range(int(count)))

def vec_to_values(vec):
    return tuple(face for face, count in enumerate(vec, start=1) for _ in range(int(count)))

def roll_label(dice_idx):
    return dice_idx_to_values(int(dice_idx))

def state_level(state):
    return int(state.filled_mask).bit_count()

def describe_state(state):
    return {
        "filled_mask": f"{state.filled_mask:013b}",
        "level": state_level(state),
        "filled": categories_from_mask(state.filled_mask),
        "upper_total": state.upper_total,
        "yahtzee_eligible": bool(state.yahtzee_eligible),
    }

## Load the value-iteration payload for one state

Each shard file contains all reduced states with the same `filled_mask`.
Rows are ordered by `(upper_total, yahtzee_eligible)`, matching how
`process_mask` sorted the states before saving.

In [3]:
def load_payload_for_state(state):
    level = state_level(state)
    path = Path(shard_path(level, state.filled_mask))
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Run value_iteration for level {level}, "
            "or check that this notebook is running from the project root."
        )
    return np.load(path)

def row_index_for_state(payload, state):
    try:
        return row_index(payload, int(state.upper_total), bool(state.yahtzee_eligible))
    except KeyError as e:
        sample = list(zip(payload["upper_total"][:10].tolist(),
                          payload["yahtzee_eligible"][:10].tolist()))
        raise KeyError(f"{e}. Available rows include: {sample} ...") from None

def get_state_row(state):
    payload = load_payload_for_state(state)
    row = row_index_for_state(payload, state)
    return payload, row

def state_value(state):
    payload, row = get_state_row(state)
    return float(payload["V"][row])

## Aggregate property helpers

These helpers inspect the forward/backward aggregate arrays we added to
`data/state_properties`.

Conventions:

- `score_dist_before` is **unnormalized**: row mass equals `reach_prob`.
- `score_dist_after` is conditional on starting from that state: row mass is about `1`.
- `box_score_dist_before_XX` is unnormalized and has mass `reach_prob` only if that box is already filled.
- `box_score_dist_after_XX` has mass `1` only if that box is still unfilled; if already filled, it is all zeros.


In [4]:

def _resolve_category(category):
    if isinstance(category, str):
        return CATEGORY_NAMES.index(category)
    return int(category)

def box_before_name(category):
    c = _resolve_category(category)
    return f"box_score_dist_before_{c:02d}"

def box_after_name(category):
    c = _resolve_category(category)
    return f"box_score_dist_after_{c:02d}"

def available_arrays(state):
    payload, row = get_state_row(state)
    return list(payload.files)

def has_array(state, name):
    payload, row = get_state_row(state)
    return name in payload.files

def get_state_array_value(state, name):
    payload, row = get_state_row(state)
    if name not in payload.files:
        raise KeyError(f"{name!r} is not in this shard. Available arrays: {payload.files}")
    return payload[name][row]

def _normalize_distribution(dist, mass=None):
    dist = np.asarray(dist, dtype=np.float64)
    if mass is None:
        mass = float(dist.sum())
    if mass <= 0:
        return dist.copy(), float(mass)
    return dist / mass, float(mass)

def distribution_stats(dist, *, mass=None, normalize=True):
    """
    Return basic stats for a discrete distribution over integer scores.

    If normalize=True, stats are computed after normalizing by mass.
    The returned 'mass' is the original row mass.
    """
    dist = np.asarray(dist, dtype=np.float64)
    raw_mass = float(dist.sum()) if mass is None else float(mass)

    if normalize:
        probs, raw_mass = _normalize_distribution(dist, raw_mass)
    else:
        probs = dist

    if probs.sum() <= 0:
        return {
            "mass": raw_mass,
            "mean": np.nan,
            "sd": np.nan,
            "q05": np.nan,
            "q25": np.nan,
            "median": np.nan,
            "q75": np.nan,
            "q95": np.nan,
            "min_nonzero": np.nan,
            "max_nonzero": np.nan,
            "n_nonzero": 0,
        }

    xs = np.arange(len(probs), dtype=np.float64)
    prob_mass = float(probs.sum())
    mean = float(xs @ probs / prob_mass)
    var = float(((xs - mean) ** 2) @ probs / prob_mass)
    cdf = np.cumsum(probs) / prob_mass

    def q(p):
        return int(np.searchsorted(cdf, p, side="left"))

    nz = np.flatnonzero(probs > 0)
    return {
        "mass": raw_mass,
        "mean": mean,
        "sd": float(np.sqrt(max(var, 0.0))),
        "q05": q(0.05),
        "q25": q(0.25),
        "median": q(0.50),
        "q75": q(0.75),
        "q95": q(0.95),
        "min_nonzero": int(nz[0]) if len(nz) else np.nan,
        "max_nonzero": int(nz[-1]) if len(nz) else np.nan,
        "n_nonzero": int(len(nz)),
    }

def distribution_table(dist, *, mass=None, normalize=True, min_prob=1e-8):
    dist = np.asarray(dist, dtype=np.float64)
    if normalize:
        probs, raw_mass = _normalize_distribution(dist, mass)
    else:
        probs = dist
        raw_mass = float(dist.sum()) if mass is None else float(mass)

    rows = [
        {"score": int(i), "prob": float(p)}
        for i, p in enumerate(probs)
        if p >= min_prob
    ]
    df = pd.DataFrame(rows)
    if len(df):
        df["cdf"] = df["prob"].cumsum()
    return df

def score_distribution(state, when="after", *, conditional=True):
    """
    Return score distribution for a state.

    when="before":
        Uses score_dist_before. This is stored unnormalized; if conditional=True,
        divide by reach_prob.

    when="after":
        Uses score_dist_after. This is already conditional on starting at state.
    """
    when = when.lower()
    payload, row = get_state_row(state)

    if when in {"before", "pre"}:
        name = "score_dist_before"
        if name not in payload.files:
            raise KeyError(f"Missing {name}. Run aggregate_properties.py forward-score-dist.")
        dist = payload[name][row].astype(np.float64)
        mass = float(payload["reach_prob"][row]) if "reach_prob" in payload.files else float(dist.sum())
        if conditional:
            return _normalize_distribution(dist, mass)[0]
        return dist

    if when in {"after", "future", "post"}:
        name = "score_dist_after"
        if name not in payload.files:
            raise KeyError(f"Missing {name}. Run aggregate_properties.py backward-score-dist.")
        return payload[name][row].astype(np.float64)

    raise ValueError("when must be 'before' or 'after'")

def score_distribution_stats(state, when="after", *, conditional=True):
    dist = score_distribution(state, when=when, conditional=conditional)
    return distribution_stats(dist)

def score_distribution_table(state, when="after", *, conditional=True, min_prob=1e-8):
    dist = score_distribution(state, when=when, conditional=conditional)
    return distribution_table(dist, min_prob=min_prob)

def box_distribution(state, category, when="after", *, conditional=True):
    """
    Return per-box distribution for a state/category.

    when="before":
        Stored as P(reach state and box has score x). If conditional=True,
        divides by reach_prob. If the box is unfilled in this state, this will
        be the all-zero distribution.

    when="after":
        Stored as future distribution for that box if it is still unfilled.
        If the box is already filled, this will be the all-zero distribution.
    """
    c = _resolve_category(category)
    when = when.lower()
    payload, row = get_state_row(state)

    if when in {"before", "pre"}:
        name = box_before_name(c)
        if name not in payload.files:
            raise KeyError(f"Missing {name}. Run aggregate_properties.py forward-box-dist --category {CATEGORY_NAMES[c]}.")
        dist = payload[name][row].astype(np.float64)
        if conditional:
            reach = float(payload["reach_prob"][row]) if "reach_prob" in payload.files else float(dist.sum())
            return _normalize_distribution(dist, reach)[0]
        return dist

    if when in {"after", "future", "post"}:
        name = box_after_name(c)
        if name not in payload.files:
            raise KeyError(f"Missing {name}. Run aggregate_properties.py backward-box-dist --category {CATEGORY_NAMES[c]}.")
        return payload[name][row].astype(np.float64)

    raise ValueError("when must be 'before' or 'after'")

def box_distribution_stats(state, category, when="after", *, conditional=True):
    dist = box_distribution(state, category, when=when, conditional=conditional)
    stats = distribution_stats(dist)
    stats["category"] = CATEGORY_NAMES[_resolve_category(category)]
    stats["when"] = when
    stats["filled_in_state"] = bool(state.filled_mask & (1 << _resolve_category(category)))
    return stats

def box_distribution_table(state, category, when="after", *, conditional=True, min_prob=1e-8):
    dist = box_distribution(state, category, when=when, conditional=conditional)
    return distribution_table(dist, min_prob=min_prob)

def state_property_summary(state):
    """
    Compact table of scalar and distribution summaries for one reduced state.
    """
    payload, row = get_state_row(state)
    rows = []

    def add(name, value):
        rows.append({"property": name, "value": value})

    add("level", state_level(state))
    add("filled", ", ".join(categories_from_mask(state.filled_mask)))
    add("upper_total", int(state.upper_total))
    add("yahtzee_eligible", bool(state.yahtzee_eligible))

    for name in [
        "V",
        "reach_prob",
        "score_sum_before",
        "expected_score_before",
        "expected_score_after_check",
        "p_top_bonus_after",
    ]:
        if name in payload.files:
            add(name, float(payload[name][row]))

    if "score_dist_before" in payload.files:
        dist = payload["score_dist_before"][row]
        reach = float(payload["reach_prob"][row]) if "reach_prob" in payload.files else float(np.sum(dist))
        stats = distribution_stats(dist, mass=reach)
        add("score_dist_before_mass", stats["mass"])
        add("score_dist_before_mean_conditional", stats["mean"])
        add("score_dist_before_median_conditional", stats["median"])

    if "score_dist_after" in payload.files:
        stats = distribution_stats(payload["score_dist_after"][row])
        add("score_dist_after_mass", stats["mass"])
        add("score_dist_after_mean", stats["mean"])
        add("score_dist_after_median", stats["median"])

    return pd.DataFrame(rows)

def all_box_summary(state, when="after", *, conditional=True):
    rows = []
    for c, name in enumerate(CATEGORY_NAMES):
        arr_name = box_after_name(c) if when.lower() in {"after", "future", "post"} else box_before_name(c)
        payload, row = get_state_row(state)
        if arr_name not in payload.files:
            continue
        stats = box_distribution_stats(state, c, when=when, conditional=conditional)
        rows.append(stats)
    return pd.DataFrame(rows)


## Inspect decisions for a specific state and roll

In [5]:
def inspect_roll(state, roll):
    '''
    roll can be a raw dice tuple/list like [1, 1, 3, 5, 6],
    or a dice_idx.
    '''
    dice_idx = int(roll) if isinstance(roll, (int, np.integer)) else dice_values_to_idx(roll)
    payload, row = get_state_row(state)

    keep_A = int(payload["decisions_A"][row, dice_idx])
    keep_B = int(payload["decisions_B"][row, dice_idx])
    cat_C = int(payload["decisions_C"][row, dice_idx])

    return pd.DataFrame([
        {
            "stage": "A: before first reroll",
            "roll": roll_label(dice_idx),
            "best_action": f"keep {keep_to_values(keep_A)}",
            "action_raw": ALL_KEEPS[keep_A],
            "EV": float(payload["ev_A"][row, dice_idx]),
        },
        {
            "stage": "B: before second reroll",
            "roll": roll_label(dice_idx),
            "best_action": f"keep {keep_to_values(keep_B)}",
            "action_raw": ALL_KEEPS[keep_B],
            "EV": float(payload["ev_B"][row, dice_idx]),
        },
        {
            "stage": "C: choose category",
            "roll": roll_label(dice_idx),
            "best_action": cat_name(cat_C),
            "action_raw": cat_C,
            "EV": float(payload["ev_C"][row, dice_idx]),
        },
    ])

## Rank first-keep or second-keep alternatives for a roll

This recomputes the EV of each legal keep from the stored downstream EV table.
For stage A, the downstream table is `ev_B`; for stage B, it is `ev_C`.

In [6]:
def keep_alternatives(state, roll, stage="A"):
    dice_idx = int(roll) if isinstance(roll, (int, np.integer)) else dice_values_to_idx(roll)
    payload, row = get_state_row(state)

    if stage.upper() == "A":
        downstream = payload["ev_B"][row]
    elif stage.upper() == "B":
        downstream = payload["ev_C"][row]
    else:
        raise ValueError("stage must be 'A' or 'B'")

    rows = []
    for keep_idx in KEEPS_FOR_DICE[dice_idx]:
        finals, nums = REROLL_OUTCOMES[(dice_idx, int(keep_idx))]
        ev = sum(downstream[fi] * n for fi, n in zip(finals, nums)) / 7776.0
        rows.append({
            "keep_idx": int(keep_idx),
            "keep": keep_to_values(keep_idx),
            "keep_vec": ALL_KEEPS[int(keep_idx)],
            "EV": float(ev),
        })

    df = pd.DataFrame(rows).sort_values("EV", ascending=False).reset_index(drop=True)
    df["EV_gap_from_best"] = df["EV"].iloc[0] - df["EV"]
    return df

## Rank final category alternatives for a roll

This mirrors the logic in `_stage_C`, but for a single state and roll,
so you can see why the category choice was made.

In [7]:
def legal_categories_for_state_and_roll(state, dice_idx):
    return state.legal_categories_by_idx(dice_idx)

def category_alternatives(state, roll):
    dice_idx = int(roll) if isinstance(roll, (int, np.integer)) else dice_values_to_idx(roll)
    payload, row = get_state_row(state)

    # Load next-level V values by mask. Missing next states only happen at
    # terminal, where continuation value is zero.
    next_level = state_level(state) + 1
    V_next_by_mask = load_V_next(next_level)

    is_joker, categories = legal_categories_for_state_and_roll(state, dice_idx)
    score_row = JOKER_SCORE_ROWS[dice_idx] if is_joker else SCORE_ROWS[dice_idx]

    rows = []
    for c in categories:
        points = int(score_row[c])
        reward = points

        if c <= SIXES:
            new_upper = min(state.upper_total + points, UPPER_BONUS_THRESHOLD)
            if state.upper_total < UPPER_BONUS_THRESHOLD and state.upper_total + points >= UPPER_BONUS_THRESHOLD:
                reward += UPPER_BONUS
        else:
            new_upper = state.upper_total

        if is_joker and state.yahtzee_eligible:
            reward += EXTRA_YAHTZEE_BONUS

        if c == YAHTZEE and points == YAHTZEE_POINTS:
            new_eligible = True
        else:
            new_eligible = bool(state.yahtzee_eligible)

        new_mask = state.filled_mask | (1 << c)
        continuation = 0.0
        if new_mask in V_next_by_mask:
            continuation = float(V_next_by_mask[new_mask][new_upper, int(new_eligible)])

        rows.append({
            "category": cat_name(c),
            "category_idx": c,
            "is_joker": bool(is_joker),
            "score_points": points,
            "immediate_reward": reward,
            "new_upper": new_upper,
            "new_eligible": new_eligible,
            "continuation_EV": continuation,
            "total_EV": reward + continuation,
        })

    df = pd.DataFrame(rows).sort_values("total_EV", ascending=False).reset_index(drop=True)
    df["EV_gap_from_best"] = df["total_EV"].iloc[0] - df["total_EV"]
    return df

In [8]:
def all_roll_evs(state, stage="A", sort=True):
    """
    Return one row per unique dice combo for a given ReducedGameState.

    stage:
        "beginning" or "state" : EV before any roll, i.e. V(state)
                                 repeated only as summary-ish context
        "A"                   : EV after seeing the initial roll,
                                 before choosing first keep
        "B"                   : EV after seeing the second roll,
                                 before choosing second keep
        "C"                   : EV after seeing the final roll,
                                 before choosing category

    For A/B/C, the EV column is pulled directly from the stored value_iteration
    payload: ev_A, ev_B, or ev_C.
    """
    stage_key = stage.lower()
    payload, row = get_state_row(state)

    if stage_key in {"beginning", "start", "state", "v"}:
        return pd.DataFrame([{
            "stage": "beginning",
            "state_EV": float(payload["V"][row]),
            "filled": categories_from_mask(state.filled_mask),
            "upper_total": state.upper_total,
            "yahtzee_eligible": bool(state.yahtzee_eligible),
        }])

    if stage_key == "a":
        evs = payload["ev_A"][row]
        decisions = payload["decisions_A"][row]
        action_type = "keep"
    elif stage_key == "b":
        evs = payload["ev_B"][row]
        decisions = payload["decisions_B"][row]
        action_type = "keep"
    elif stage_key == "c":
        evs = payload["ev_C"][row]
        decisions = payload["decisions_C"][row]
        action_type = "category"
    else:
        raise ValueError("stage must be one of: 'beginning', 'A', 'B', or 'C'")

    rows = []
    for dice_idx in range(len(ALL_DICE_STATES)):
        decision = int(decisions[dice_idx])

        if action_type == "keep":
            best_action = keep_to_values(decision)
            best_action_raw = ALL_KEEPS[decision]
        else:
            best_action = cat_name(decision)
            best_action_raw = decision

        rows.append({
            "stage": stage.upper(),
            "dice_idx": dice_idx,
            "roll": roll_label(dice_idx),
            "roll_vec": tuple(int(x) for x in ALL_DICE_STATES[dice_idx]),
            "roll_freq": int(ALL_DICE_FREQS[dice_idx]),
            "probability": float(ALL_DICE_FREQS[dice_idx] / 7776),
            "best_action": best_action,
            "best_action_raw": best_action_raw,
            "EV": float(evs[dice_idx]),
        })

    df = pd.DataFrame(rows)
    if sort:
        df = df.sort_values("EV", ascending=False).reset_index(drop=True)
    return df

## Find interesting rolls in a state

This lists rolls where the optimal first keep is close to another option,
which is often useful for debugging strategy behavior.

In [9]:
def closest_first_keep_margins(state, n=20):
    rows = []
    for dice_idx in range(len(ALL_DICE_STATES)):
        alts = keep_alternatives(state, dice_idx, stage="A")
        if len(alts) < 2:
            continue
        rows.append({
            "roll": roll_label(dice_idx),
            "best_keep": alts.loc[0, "keep"],
            "best_EV": alts.loc[0, "EV"],
            "second_keep": alts.loc[1, "keep"],
            "second_EV": alts.loc[1, "EV"],
            "margin": alts.loc[0, "EV"] - alts.loc[1, "EV"],
        })
    return pd.DataFrame(rows).sort_values("margin").head(n).reset_index(drop=True)

## Example usage

Edit these examples to match the state you care about.

In [10]:
# Example: after Ones and Twos have been filled, with upper_total=6,
# and no scored Yahtzee yet.
state = ReducedGameState(
    filled_mask=mask_from_categories(["Ones", "Twos"]),
    upper_total=6,
    yahtzee_eligible=False,
)

describe_state(state), state_value(state)

({'filled_mask': '0000000000011',
  'level': 2,
  'filled': ['Ones', 'Twos'],
  'upper_total': 6,
  'yahtzee_eligible': False},
 223.743408203125)

In [11]:
inspect_roll(state, [1, 1, 3, 5, 6])

,stage,roll,best_action,action_raw,EV
0,A: before first reroll,"(1, 1, 3, 5, 6)","keep (5,)","(0, 0, 0, 0, 1, 0)",219.093491
1,B: before second reroll,"(1, 1, 3, 5, 6)","keep (3, 5, 6)","(0, 0, 1, 0, 1, 1)",213.797455
2,C: choose category,"(1, 1, 3, 5, 6)",Chance,11,205.569092


In [12]:
keep_alternatives(state, [1, 1, 3, 5, 6], stage="A")

,keep_idx,keep,keep_vec,EV,EV_gap_from_best
0,6,"(5,)","(0, 0, 0, 0, 1, 0)",219.093475,0.000000
1,61,"(3, 5)","(0, 0, 1, 0, 1, 0)",219.055099,0.038376
2,1,"(6,)","(0, 0, 0, 0, 0, 1)",219.000534,0.092941
3,56,"(3,)","(0, 0, 1, 0, 0, 0)",218.839874,0.253601
4,7,"(5, 6)","(0, 0, 0, 0, 1, 1)",218.754028,0.339447
5,57,"(3, 6)","(0, 0, 1, 0, 0, 1)",218.520416,0.573059
6,0,(),"(0, 0, 0, 0, 0, 0)",218.417938,0.675537
7,62,"(3, 5, 6)","(0, 0, 1, 0, 1, 1)",218.400970,0.692505
8,287,"(1, 3)","(1, 0, 1, 0, 0, 0)",217.150345,1.943130
9,257,"(1, 5)","(1, 0, 0, 0, 1, 0)",217.078354,2.015121


In [13]:
keep_alternatives(state, [1, 1, 3, 5, 6], stage="B")

,keep_idx,keep,keep_vec,EV,EV_gap_from_best
0,62,"(3, 5, 6)","(0, 0, 1, 0, 1, 1)",213.797455,0.000000
1,61,"(3, 5)","(0, 0, 1, 0, 1, 0)",213.585526,0.211929
2,7,"(5, 6)","(0, 0, 0, 0, 1, 1)",213.524841,0.272614
3,56,"(3,)","(0, 0, 1, 0, 0, 0)",213.327637,0.469818
4,6,"(5,)","(0, 0, 0, 0, 1, 0)",213.105560,0.691895
5,1,"(6,)","(0, 0, 0, 0, 0, 1)",212.875229,0.922226
6,57,"(3, 6)","(0, 0, 1, 0, 0, 1)",212.853394,0.944061
7,0,(),"(0, 0, 0, 0, 0, 0)",212.330872,1.466583
8,287,"(1, 3)","(1, 0, 1, 0, 0, 0)",211.079025,2.718430
9,291,"(1, 3, 5)","(1, 0, 1, 0, 1, 0)",210.328094,3.469360


In [14]:
category_alternatives(state, [1, 1, 3, 5, 6])

,category,category_idx,is_joker,score_points,immediate_reward,new_upper,new_eligible,continuation_EV,total_EV,EV_gap_from_best
0,Chance,11,False,16,16,6,False,189.569092,205.569092,0.000000
1,4Kind,7,False,0,0,6,False,205.363449,205.363449,0.205643
2,Threes,2,False,3,3,9,False,201.207718,204.207718,1.361374
3,Yahtzee,12,False,0,0,6,False,202.948227,202.948227,2.620865
4,Fives,4,False,5,5,11,False,195.272995,200.272995,5.296097
5,Sixes,5,False,6,6,12,False,193.255951,199.255951,6.313141
6,FullHouse,8,False,0,0,6,False,199.253510,199.253510,6.315582
7,Fours,3,False,0,0,6,False,197.216339,197.216339,8.352753
8,3Kind,6,False,0,0,6,False,195.767609,195.767609,9.801483
9,LgStraight,10,False,0,0,6,False,192.148941,192.148941,13.420151


In [15]:
closest_first_keep_margins(state, n=20)

,roll,best_keep,best_EV,second_keep,second_EV,margin
0,"(1, 2, 3, 5, 6)","(5,)",219.093475,"(3, 5)",219.055099,0.038376
1,"(1, 2, 2, 3, 5)","(5,)",219.093475,"(3, 5)",219.055099,0.038376
2,"(2, 2, 3, 5, 6)","(5,)",219.093475,"(3, 5)",219.055099,0.038376
3,"(1, 1, 2, 3, 5)","(5,)",219.093475,"(3, 5)",219.055099,0.038376
4,"(1, 1, 3, 5, 6)","(5,)",219.093475,"(3, 5)",219.055099,0.038376
5,"(1, 2, 2, 5, 6)","(5,)",219.093475,"(6,)",219.000534,0.092941
6,"(1, 1, 2, 5, 6)","(5,)",219.093475,"(6,)",219.000534,0.092941
7,"(1, 2, 2, 4, 6)","(4,)",219.103531,"(6,)",219.000534,0.102997
8,"(1, 1, 2, 4, 6)","(4,)",219.103531,"(6,)",219.000534,0.102997
9,"(1, 2, 4, 5, 6)","(4, 5)",219.242813,"(4,)",219.103531,0.139282


In [16]:
start_state = ReducedGameState(
    filled_mask=mask_from_categories([]),
    upper_total=0,
    yahtzee_eligible=False,
)

describe_state(start_state), state_value(start_state)

({'filled_mask': '0000000000000',
  'level': 0,
  'filled': [],
  'upper_total': 0,
  'yahtzee_eligible': False},
 254.5877227783203)

In [17]:
a = 7
all_roll_evs(start_state, "C")[200:]

,stage,dice_idx,roll,roll_vec,roll_freq,probability,best_action,best_action_raw,EV
200,C,92,"(2, 2, 5, 6, 6)","(0, 2, 0, 0, 1, 2)",30,0.003858,Twos,1,241.752762
201,C,182,"(1, 2, 2, 5, 6)","(1, 2, 0, 0, 1, 1)",60,0.007716,Twos,1,241.752762
202,C,181,"(1, 2, 2, 6, 6)","(1, 2, 0, 0, 0, 2)",30,0.003858,Twos,1,241.752762
203,C,187,"(1, 2, 2, 3, 6)","(1, 2, 1, 0, 0, 1)",60,0.007716,Twos,1,241.752762
204,C,183,"(1, 2, 2, 5, 5)","(1, 2, 0, 0, 2, 0)",30,0.003858,Twos,1,241.752762
205,C,184,"(1, 2, 2, 4, 6)","(1, 2, 0, 1, 0, 1)",60,0.007716,Twos,1,241.752762
206,C,190,"(1, 2, 2, 3, 3)","(1, 2, 2, 0, 0, 0)",30,0.003858,Twos,1,241.752762
207,C,185,"(1, 2, 2, 4, 5)","(1, 2, 0, 1, 1, 0)",60,0.007716,Twos,1,241.752762
208,C,186,"(1, 2, 2, 4, 4)","(1, 2, 0, 2, 0, 0)",30,0.003858,Twos,1,241.752762
209,C,188,"(1, 2, 2, 3, 5)","(1, 2, 1, 0, 1, 0)",60,0.007716,Twos,1,241.752762


In [18]:
"""CATEGORY_NAMES = [
    "Ones", "Twos", "Threes", "Fours", "Fives", "Sixes",
    "3Kind", "4Kind", "FullHouse",
    "SmStraight", "LgStraight", "Chance", "Yahtzee",
]"""

'CATEGORY_NAMES = [\n    "Ones", "Twos", "Threes", "Fours", "Fives", "Sixes",\n    "3Kind", "4Kind", "FullHouse",\n    "SmStraight", "LgStraight", "Chance", "Yahtzee",\n]'

In [19]:
missing = 0
cats = ["Ones", "Twos", "Threes", "Fours", "Fives", "Sixes", "3Kind", "4Kind", "FullHouse", "SmStraight", "LgStraight", "Chance", "Yahtzee"]
reduced_cats = cats[:missing] + cats[missing + 1:]
near_end_state = ReducedGameState(
    filled_mask=mask_from_categories(reduced_cats),
    upper_total=58,
    yahtzee_eligible=True,
)

describe_state(near_end_state), state_value(near_end_state)

({'filled_mask': '1111111111110',
  'level': 12,
  'filled': ['Twos',
   'Threes',
   'Fours',
   'Fives',
   'Sixes',
   '3Kind',
   '4Kind',
   'FullHouse',
   'SmStraight',
   'LgStraight',
   'Chance',
   'Yahtzee'],
  'upper_total': 58,
  'yahtzee_eligible': True},
 6.110448360443115)

In [20]:
all_roll_evs(near_end_state, "B")

,stage,dice_idx,roll,roll_vec,roll_freq,probability,best_action,best_action_raw,EV
0,B,251,"(1, 1, 1, 1, 1)","(5, 0, 0, 0, 0, 0)",1,0.000129,"(1, 1, 1, 1, 1)","(5, 0, 0, 0, 0, 0)",140.000000
1,B,5,"(5, 5, 5, 5, 5)","(0, 0, 0, 0, 5, 0)",1,0.000129,"(5, 5, 5, 5, 5)","(0, 0, 0, 0, 5, 0)",100.000000
2,B,125,"(2, 2, 2, 2, 2)","(0, 5, 0, 0, 0, 0)",1,0.000129,"(2, 2, 2, 2, 2)","(0, 5, 0, 0, 0, 0)",100.000000
3,B,55,"(3, 3, 3, 3, 3)","(0, 0, 5, 0, 0, 0)",1,0.000129,"(3, 3, 3, 3, 3)","(0, 0, 5, 0, 0, 0)",100.000000
4,B,20,"(4, 4, 4, 4, 4)","(0, 0, 0, 5, 0, 0)",1,0.000129,"(4, 4, 4, 4, 4)","(0, 0, 0, 5, 0, 0)",100.000000
5,B,0,"(6, 6, 6, 6, 6)","(0, 0, 0, 0, 0, 5)",1,0.000129,"(6, 6, 6, 6, 6)","(0, 0, 0, 0, 0, 5)",100.000000
6,B,250,"(1, 1, 1, 1, 2)","(4, 1, 0, 0, 0, 0)",5,0.000643,"(1, 1, 1, 1)","(4, 0, 0, 0, 0, 0)",26.666666
7,B,249,"(1, 1, 1, 1, 3)","(4, 0, 1, 0, 0, 0)",5,0.000643,"(1, 1, 1, 1)","(4, 0, 0, 0, 0, 0)",26.666666
8,B,248,"(1, 1, 1, 1, 4)","(4, 0, 0, 1, 0, 0)",5,0.000643,"(1, 1, 1, 1)","(4, 0, 0, 0, 0, 0)",26.666666
9,B,247,"(1, 1, 1, 1, 5)","(4, 0, 0, 0, 1, 0)",5,0.000643,"(1, 1, 1, 1)","(4, 0, 0, 0, 0, 0)",26.666666


## Aggregate property examples

These examples assume you have run the aggregate scripts. They are safe to edit
for whatever state you are studying.


In [21]:

# State-level scalar and score-distribution summary.
state_property_summary(start_state)


,property,value
0,level,0
1,filled,
2,upper_total,0
3,yahtzee_eligible,False
4,V,254.587723


In [22]:

# Future score distribution from the start state.
score_distribution_table(start_state, when="after", min_prob=0.005)


KeyError: 'Missing score_dist_after. Run aggregate_properties.py backward-score-dist.'

In [23]:

# Per-box future distributions from the start state.
all_box_summary(start_state, when="after")


""


In [24]:

# Example: Full House future distribution from the start state.
box_distribution_table(start_state, "FullHouse", when="after", min_prob=1e-6)


KeyError: 'Missing box_score_dist_after_08. Run aggregate_properties.py backward-box-dist --category FullHouse.'